# 🚀 통합 농산물 가격 예측 하이브리드 모델링
본 노트북은 ML(LGBM)과 DL(Chronos-2 Large)을 결합하여 21개 농작물의 t+7, t+14, t+28 가격을 예측합니다.

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from datetime import timedelta
import lightgbm as lgb
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import optuna
import torch
from korean_font import set_korean_font


# 구글 코랩 환경 확인 및 드라이브 마운트
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/data/' # 사용자 경로에 맞게 수정
    # Colab 한글 폰트 설정
    !sudo apt-get -qq -y install fonts-nanum > /dev/null
    import matplotlib.font_manager as fm
    font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
    fm.fontManager.addfont(font_path)
    plt.rc('font', family='NanumBarunGothic')
else:
    DATA_PATH = 'data/'
    plt.rc('font', family='Malgun Gothic') # Windows


set_korean_font()

plt.rcParams['axes.unicode_minus'] = False
print("환경 설정 완료.")

## 2. 데이터 탐색 (EDA) 및 시각화

In [ ]:
from preprocess import preprocess
# 데이터 로드
train_raw = pd.read_csv(DATA_PATH + 'train.csv')
test_raw = pd.read_csv(DATA_PATH + 'test.csv')

train_raw = preprocess(train_raw)
test_raw = preprocess(test_raw)


In [ ]:
train_raw.head()

In [ ]:
train_raw.info()

In [ ]:
# 농산물 목록 추출
crops = train_raw['품목'].unique().tolist()
print(f"총 {len(crops)}개 품목: {crops}")


In [ ]:
# 1. 데이터를 피벗 테이블로 변환 (행: 날짜, 열: 품목, 값: 평균가격)
pivot_df = train_raw.pivot(index='DATE', columns='품목', values='평균가격')
# 2. 상관계수 계산
corr_matrix = pivot_df.corr()
# 3. 히트맵 그리기
plt.figure(figsize=(15, 12))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('품목별 평균가격 상관관계')
plt.show()

## 3. 데이터 제약, 전처리 및 파생 변수 생성

In [ ]:
# 병합하여 전처리 수행
combined_df = pd.concat([train_raw, test_raw], axis=0).sort_values('date').reset_index(drop=True)
combined_df = combined_df.set_index('date').asfreq('D')

# 결측치 처리 (보간 - Forward Fill)
combined_df = combined_df.ffill().fillna(0)

# 파생 변수: is_season 생성 (거래 유무 파악)
for crop in crops:
    combined_df['_is_season'] = (combined_df['총거래물량'] > 0).astype(int)

# 타겟 변수 변환: np.log1p (저가 품목 오차율 통제)
# 원본 가격 저장을 위해 복사본 생성 (NMAE 계산용)
combined_df_orig = combined_df.copy()

combined_df['총거래금액'] = np.log1p(combined_df['총거래금액'])
combined_df['총거래물량'] = np.log1p(combined_df['총거래물량'])
combined_df['평균가격'] = np.log1p(combined_df['평균가격'])

# 명절 특성 추가
holidays = pd.to_datetime(['2022-02-01', '2022-09-10', '2023-01-22', '2023-09-29', 
                           '2024-02-10', '2024-09-17', '2025-01-29', '2025-10-06', '2026-02-17'])

# 초기값을 0이 아닌 특이값으로 설정하여 평시와 명절 당일을 구분
combined_df['holiday'] = 0
combined_df['holiday_dday'] = 999  # 평시를 의미하는 Out-of-range 값

for h in holidays:
    start = h - pd.Timedelta(days=14)
    end = h + pd.Timedelta(days=7)
    mask = (combined_df.index >= start) & (combined_df.index <= end)
    combined_df.loc[mask, 'holiday'] = 1
    # 기존 값보다 더 절댓값이 작은(명절에 더 가까운) 경우에만 업데이트 (중첩 방지)
    current_dday = (combined_df.index[mask] - h).days
    combined_df.loc[mask, 'holiday_dday'] = (combined_df.index[mask] - h).days
    
combined_df = combined_df.sort_values(['품목', 'DATE'])

# 1. Lag 변수 생성 (품목별로 격리하여 shift)
for lag in [1, 2, 3, 7, 14]:
    combined_df[f'lag_{lag}'] = combined_df.groupby('품목')['총거래금액'].transform(lambda x: x.shift(lag))

# 2. 이동평균 및 표준편차 생성
for window in [7, 14, 28]:
    group_obj = combined_df.groupby('품목')['총거래금액']
    combined_df[f'ma_{window}'] = group_obj.transform(lambda x: x.rolling(window).mean())
    combined_df[f'std_{window}'] = group_obj.transform(lambda x: x.rolling(window).std())

combined_df = combined_df.dropna()
print("데이터 전처리 완료. shape:", combined_df.shape)

#### 유가 데이터 추가

In [ ]:
oil = pd.read_csv('data/oil_price.csv')
combined_df = pd.merge(combined_df, oil, on='DATE', how='left')

# 3. 결측치 확인 및 처리
# 만약 기존 데이터의 날짜가 유가 데이터 범위를 벗어날 경우 NaN이 발생할 수 있음
nan_train = train_raw['Oil_Price'].isna().sum()
nan_test = test_raw['Oil_Price'].isna().sum()

if nan_train > 0 or nan_test > 0:
    print(f"⚠️ 결측치 발생: Train {nan_train}건, Test {nan_test}건")
    # 결측치가 있다면 가장 가까운 값으로 채움 (앞뒤 방향)
    train_raw['Oil_Price'] = train_raw['Oil_Price'].ffill().bfill()
    test_raw['Oil_Price'] = test_raw['Oil_Price'].ffill().bfill()

# 4. 결과 확인
print("✅ 결합 완료")
print(f"Train columns: {train_raw.columns.tolist()}")
print(f"Test columns: {test_raw.columns.tolist()}")
print("-" * 30)
print(f"Train Oil_Price Sample:\n{train_raw[['Date', 'Oil_Price']].head()}")

In [ ]:
# 28일 전 유가 정보 생성 (28일 후 가격 예측 시 '미래의 알려진 정보'로 쓰임)
combined_df['Oil_Price_lag28'] = combined_df.groupby('품목')['Oil_Price'].shift(28)

# 결측치는 가까운 값으로 채움
combined_df['Oil_Price_lag28'] = combined_df['Oil_Price_lag28'].ffill().bfill()


#### 날씨 데이터 추가

In [ ]:
def get_weather_flags(df_weather):
    df_w = df_weather.copy()
    df_w['date'] = pd.to_datetime(df_w['DATE'])
    df_w['day_of_year'] = df_w['DATE'].dt.dayofyear
    
    # 통계적 기준(mu, sigma) 계산
    baseline = df_w.groupby(['location', 'day_of_year'])['avg_temp'].agg(['mean', 'std']).reset_index()
    baseline.columns = ['location', 'day_of_year', 'temp_mu', 'temp_sigma']
    df_w = df_w.merge(baseline, on=['location', 'day_of_year'], how='left')

    # --- 위험 플래그 생성 ---
    # 1. 이상 고온/저온 (통계적 1.5 시그마 일탈)
    df_w['flag_extreme_hot'] = ((df_w['avg_temp'] - df_w['temp_mu']) > 1.5 * df_w['temp_sigma']).astype(int)
    df_w['flag_extreme_cold'] = ((df_w['avg_temp'] - df_w['temp_mu']) < -1.5 * df_w['temp_sigma']).astype(int)
    
    # 2. 절대적 위험 (폭염, 한파, 폭우)
    df_w['flag_heatwave'] = (df_w['max_temp'] >= 33).astype(int)
    df_w['flag_coldwave'] = (df_w['min_temp'] <= -10).astype(int)
    df_w['flag_heavy_rain'] = (df_w['precip'] >= 80).astype(int)
    
    # 3. 가뭄 (15일 누적 강수 5mm 미만)
    df_w['precip_15d'] = df_w.groupby('location')['precip'].transform(lambda x: x.rolling(15, min_periods=1).sum())
    df_w['flag_drought'] = (df_w['precip_15d'] < 5).astype(int)

    # 필요한 컬럼만 추출하여 반환
    flag_cols = ['date', 'location', 'flag_extreme_hot', 'flag_extreme_cold', 
                 'flag_heatwave', 'flag_coldwave', 'flag_heavy_rain', 'flag_drought']
    return df_w[flag_cols]

# 실행
df_weather_flags = get_weather_flags(df_weather)

In [ ]:
# 확장된 주산지 매핑
item_region_map = {
    '배추': ['평창', '해남'],       # 고랭지(평창), 겨울(해남)
    '무': ['제주', '평창'],         # 겨울(제주/해남), 여름(평창)
    '양파': ['목포', '밀양'],               # 전국 최대 양파 주산지 2곳
    '마늘': ['밀양', '목포'],               # 한지/난지형 마늘 주산지
    '대파': ['신안', '평창'],       # 전남권 대파 벨트 (신안은 대파 핵심)
    '당근': ['제주'],                       # 국내 당근 생산의 압도적 비중
    '청상추': ['부여'],             # 논산은 상추/엽채류의 메카
    '깻잎': ['상주'],
    '시금치': ['동두천', '신안'],             # 섬초 및 겨울 시금치
    '백다다기': ['상주'],           # 오이 주산지(상주)
    '애호박': ['상주', '부여'],
    '샤인마스캇': ['상주'],                 # 포도 최대 주산지 중 하나
    '캠벨얼리': ['상주'],
    '토마토': ['평창'],
    '파프리카': ['평창'],
    '양배추': ['제주', '평창', '목포'],
    '건고추': ['해남']              # 노지 채소 산지 기반
}

all_regions = ['해남', '평창', '목포', '제주', '신안', '밀양', '상주', '동두천', '부여']

In [ ]:
def merge_only_flags(df_train, df_weather_flags):
    final_dfs = []
    for item in df_train['품목'].unique():
        item_df = df_train[df_train['품목'] == item].copy()
        target_regions = item_region_map.get(item, all_regions)
        
        # 해당 주산지들 중 하나라도 위험 플래그가 1이면 1로 처리
        weather_summary = df_weather_flags[df_weather_flags['location'].isin(target_regions)]
        weather_summary = weather_summary.groupby('date').max().reset_index()
        
        # location 컬럼은 groupby 후 의미 없으므로 제거
        weather_summary = weather_summary.drop(columns=['location'])
        
        merged = item_df.merge(weather_summary, on='date', how='left').fillna(0)
        final_dfs.append(merged)
        
    return pd.concat(final_dfs, ignore_index=True)



In [ ]:
# 실행
conbined_df = merge_only_flags(df_train, df_weather_flags)
conbined_df.info()

In [ ]:
def apply_crop_specific_lags(df_final):
    """
    품목별 생리 특성에 맞는 시차 및 누적 플래그 생성
    """
    # 1. 품목 그룹 정의
    short_cycle = ['청상추', '시금치', '깻잎', '미나리', '얼갈이배추']
    mid_cycle = ['토마토', '애호박', '파프리카', '백다다기']
    long_cycle = ['배추', '무', '양파', '마늘', '당근', '양배추', '건고추']
    
    df_lagged = df_final.sort_values(['item_name', 'date']).copy()
    
    # 기본 위험 플래그 리스트
    target_flags = ['flag_heatwave', 'flag_heavy_rain', 'flag_coldwave']
    
    for item in df_lagged['item_name'].unique():
        mask = df_lagged['item_name'] == item
        
        if item in short_cycle:
            # 엽채류: 7일, 14일 시차 및 7일 누적합
            for flag in target_flags:
                df_lagged.loc[mask, f'{flag}_L7'] = df_lagged[mask][flag].shift(7)
                df_lagged.loc[mask, f'{flag}_R7'] = df_lagged[mask][flag].rolling(7).sum()
                
        elif item in mid_cycle:
            # 과채류: 14일, 21일 시차 및 14일 누적합
            for flag in target_flags:
                df_lagged.loc[mask, f'{flag}_L14'] = df_lagged[mask][flag].shift(14)
                df_lagged.loc[mask, f'{flag}_R14'] = df_lagged[mask][flag].rolling(14).sum()
                
        elif item in long_cycle:
            # 구근/저장: 30일 시차 및 30일 누적합
            for flag in target_flags:
                df_lagged.loc[mask, f'{flag}_L30'] = df_lagged[mask][flag].shift(30)
                df_lagged.loc[mask, f'{flag}_R30'] = df_lagged[mask][flag].rolling(30).sum()
                
    return df_lagged.fillna(0)

# 실행
df_final_lagged = apply_crop_specific_lags(df_final)

## 4. 품목 임베딩 사전 생성 (PCA)

In [ ]:
# 품목별 '정체성'을 나타내는 다차원 통계량 추출
crop_stats = []
train_end_date = train_raw['date'].max()
for crop in crops:
    # 해당 품목의 로그 변환된 데이터 추출
    item_data = combined_df[(combined_df['품목'] == crop) & (combined_df['date'] <= train_end_date)]
    
    # 1) 기초 통계
    # 로그 스케일에서 mean은 가격의 '자릿수(체급)'를 의미합니다.
    mean_log_price = item_data['평균가격'].mean()
    
    # 로그 스케일에서 표준편차(std)는 그 자체로 '상대적 변동률'에 근사합니다.
    # (따로 mean으로 나눌 필요 없이 std 자체가 훌륭한 변동성 지표가 됩니다.)
    volatility = item_data['평균가격'].std() 
    
    trade_freq = item_data['_is_season'].mean() # 앞선 셀에서 _is_season으로 생성됨
    
    # 2) 유가 민감도 (상관계수)
    # 로그-로그 공간에서의 상관계수는 '탄력성'을 포착하기에 좋습니다.
    oil_corr = item_data[['평균가격', 'Oil_Price']].corr().iloc[0, 1]
    
    # 3) 기상 민감도 (중요: 로그 스케일이므로 '차이'를 계산)
    # 로그(폭염가격) - 로그(평균가격) = 로그(폭염가격/평균가격) -> 즉, % 상승률을 의미
    heatwave_mask = item_data['flag_heatwave'] == 1
    if heatwave_mask.any():
        # 평소 대비 폭염 시 가격이 몇 %나 더 높은가를 나타내는 지표
        weather_impact = item_data[heatwave_mask]['평균가격'].mean() - mean_log_price
    else:
        weather_impact = 0 # 로그 스케일에서 차이가 0이면 원래 스케일에서 1배(변화 없음)
    
    crop_stats.append([volatility, trade_freq, mean_log_price, oil_corr, weather_impact])

# 스케일링 및 PCA
scaler = StandardScaler()
crop_stats_scaled = scaler.fit_transform(np.nan_to_num(crop_stats)) # 결측치 처리 포함

pca = PCA(n_components=3) # 3차원으로 품목의 성격을 압축
crop_embeddings_pca = pca.fit_transform(crop_stats_scaled)

# 모델에 입력할 수 있도록 사전화
crop_vec_dict = {crop: emb for crop, emb in zip(crops, crop_embeddings_pca)}

## 5. 모델링 파트 1: ML (LGBM) - 독립 모델 학습 및 예측

In [ ]:
# Train/Test 분리 (date 컬럼 기준)
train_df = combined_df[combined_df['date'] <= train_end_date].copy()
test_df = combined_df[combined_df['date'] > train_end_date].copy()

def get_features(df_item):
    """
    품목별 데이터프레임에서 학습에 사용할 피처 컬럼들을 동적으로 선별
    """
    # 1. 기본 공통 피처
    features = ['holiday', 'holiday_dday', '_is_season', 'Oil_Price']
    
    # 2. PCA 벡터 피처 (품목 정체성)
    features += ['vec0', 'vec1', 'vec2']
    
    # 3. 동적으로 생성된 피처들 (Lag, MA, Std, 품목별 기상 시차 변수 등)
    # 기존 코드의 의도인 lag, ma, std를 포함하여 전처리에서 생성된 모든 변수를 자동 추가
    exclude_cols = ['date', 'DATE', '품목', 'item_name', '평균가격', '총거래금액', '총거래물량']
    dynamic_features = [c for c in df_item.columns if c not in exclude_cols and c not in features]
    
    return features + dynamic_features

horizons = [7, 14, 28]
lgbm_models = {}
lgbm_preds = {}

for crop in crops:
    lgbm_models[crop] = {}
    lgbm_preds[crop] = {}
    
    # 해당 품목 데이터만 추출 (롱 포맷 대응)
    item_train = train_df[train_df['품목'] == crop].copy()
    item_test = test_df[test_df['품목'] == crop].copy()
    
    if len(item_train) == 0: continue
    
    # PCA 임베딩 정보 주입 (기존 의도 유지)
    vec = crop_vec_dict[crop]
    for _df in [item_train, item_test]:
        _df['vec0'], _df['vec1'], _df['vec2'] = vec[0], vec[1], vec[2]
        
    features = get_features(item_train)
    
    for h in horizons:
        # 타겟 설정: 로그 변환된 '평균가격'의 미래 시점 (shift -h)
        y_train = item_train['평균가격'].shift(-h).dropna()
        X_train = item_train[features].loc[y_train.index]
        
        # Test 데이터셋 구성 (NMAE 계산을 위한 실제 가격 shift 포함)
        y_test = item_test['평균가격'].shift(-h).dropna()
        X_test = item_test[features].loc[y_test.index]
        
        if len(y_test) == 0: continue
        
        # 모델 학습 (기존 파라미터 및 L1 objective 유지)
        model = lgb.LGBMRegressor(objective='regression_l1', n_estimators=150, random_state=42, verbose=-1)
        model.fit(X_train, y_train)
        
        lgbm_models[crop][h] = model
        
        # 예측 수행
        preds = model.predict(X_test)
        lgbm_preds[crop][h] = pd.Series(preds, index=y_test.index)

print("LGBM 학습 및 Test 예측 완료.")


## 6. 모델링 파트 2: DL (Chronos-2 Large) - Zero-shot 기반 문맥 추론

In [ ]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# 1. 크로노스 전용 데이터셋 형태로 변환 (Long Format 활용)
df_chronos = combined_df.reset_index().rename(columns={
    'date': 'timestamp', 
    '품목': 'item_id', 
    '평균가격': 'target'
})

# 2. 우리가 만든 수많은 다변량 피처(기상 시차, 유가 등)를 리스트업
# 타겟과 식별자를 제외한 모든 피처를 공변량(Covariates)으로 사용합니다.
covariate_cols = [c for c in df_chronos.columns if 'flag_' in c or 'holiday' in c or 'Oil_Price_lag28' in c]

# 3. TimeSeriesDataFrame 생성 
ts_data = TimeSeriesDataFrame.from_data_frame(
    df_chronos,
    id_column="item_id",
    timestamp_column="timestamp"
)

# 3. 데이터 분할
# 과거 문맥: 훈련 종료일까지
past_data = ts_data.slice_by_time(None, train_end_date)


# 미래 공변량: 예측 기간(28일) 동안의 외부 변수 정보
# ts_data에서 해당 기간의 공변량 컬럼들만 추출
future_covariates_df = ts_data.slice_by_time(
    pd.Timestamp(train_end_date) + pd.Timedelta(days=1), 
    pd.Timestamp(train_end_date) + pd.Timedelta(days=28)
)[covariate_cols]


# 4. Predictor 설정 및 추론
predictor = TimeSeriesPredictor(
    prediction_length=28, # 최대 28일까지 예측
    target="target",
    eval_metric="NMAE",
    known_covariates_names=covariate_cols
)

# 모델 추론 (Zero-shot)
# 학습(fit) 과정 없이, 준비된 Chronos-Large 모델의 지능만 빌려 바로 예측합니다.
# 'Chronos.모델명'으로 호출하면 Zero-shot 모드로 작동합니다.
chronos_predictions = predictor.predict(
    past_data, 
    known_covariates=future_covariates_df,
    model="Chronos-Large", 
    random_seed=42
)

# 5. 앙상블 셀을 위한 딕셔너리 구조 변환 (필수)
# 다음 셀에서 chronos_preds[crop][h] 형태를 사용하므로 맞춰줘야 합니다.
chronos_preds = {}
for crop in crops:
    chronos_preds[crop] = {}
    item_pred = chronos_predictions.loc[crop]
    for h in horizons:
        # h일 뒤의 예측값들을 Series 형태로 저장
        # (LGBM 예측값의 인덱스와 일치시키기 위해 데이터프레임에서 추출)
        target_date = pd.Timestamp(train_end_date) + pd.Timedelta(days=h)
        if target_date in item_pred.index:
            # 앙상블 로직에 맞게 h일 시점의 단일 값을 포함한 Series 생성
            # 실제 앙상블에서는 h 시점별로 루프를 돌기 때문에 구조를 맞춰야 함
            chronos_preds[crop][h] = item_pred.loc[target_date]['mean']
print("✅ 다변량 및 미래공변량을 반영한 Chronos Zero-shot 추론 및 구조 변환 완료.")


In [ ]:
# # Zero-shot이 아닌, 한국 농산물 데이터에 살짝 적응시키기
# predictor.fit(
#     train_data=ts_data,
#     hyperparameters={
#         "Chronos": {"model_size": "large", "max_steps": 500} # 짧게 학습
#     }
# )

## 7. 모델링 파트 3: 가중치 최적화 및 메타 앙상블

In [ ]:
import optuna
import numpy as np
import pandas as pd

def nmae(y_true, y_pred):
    # 실제 가격 기준 NMAE 계산 (0보다 큰 값만 대상)
    mask = (y_true > 0) & (np.isfinite(y_true))
    if mask.sum() == 0: return 0.0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / y_true[mask])

ensemble_weights = {}
final_preds = {}

# 로그 출력 억제
optuna.logging.set_verbosity(optuna.logging.WARNING)

for crop in crops:
    ensemble_weights[crop] = {}
    final_preds[crop] = {}
    
    # 해당 품목의 테스트 정답 데이터 필터링
    item_test = test_df[test_df['품목'] == crop].copy()
    
    for h in horizons:
        # LGBM과 Chronos 예측값이 모두 존재하는지 확인
        if h not in lgbm_preds.get(crop, {}) or h not in chronos_preds.get(crop, {}):
            continue
            
        # 1. 실제 정답값 준비 (로그 -> 지수 변환)
        # '평균가격' 컬럼명을 사용하며, h일 시차를 적용합니다.
        y_true_log = item_test['평균가격'].shift(-h).dropna()
        y_true_raw = np.expm1(y_true_log)
        
        # 2. 공통 인덱스 추출 (예측값과 정답 일치)
        common_idx = y_true_raw.index.intersection(lgbm_preds[crop][h].index)
        if len(common_idx) == 0: continue
        
        y_true = y_true_raw.loc[common_idx]
        pred_lgb = lgbm_preds[crop][h].loc[common_idx]
        pred_chr = chronos_preds[crop][h].loc[common_idx] # Series 형태여야 함
        
        # 3. Optuna 목적 함수 정의
        def objective(trial):
            # LGBM 가중치 결정 (0 ~ 1)
            w_lgb = trial.suggest_float('w_lgb', 0.0, 1.0)
            w_chr = 1.0 - w_lgb
            
            # 로그 공간에서 가중합 수행 후 지수 변환 (안정적인 결합)
            ensemble_log = w_lgb * pred_lgb + w_chr * pred_chr
            ensemble_exp = np.expm1(ensemble_log)
            
            return nmae(y_true, ensemble_exp)
            
        # 4. 최적화 실행
        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=30) # 30회 정도면 충분히 수렴
        
        best_w_lgb = study.best_params['w_lgb']
        ensemble_weights[crop][h] = best_w_lgb
        
        # 5. 최종 예측값 저장
        best_ensemble_log = best_w_lgb * pred_lgb + (1.0 - best_w_lgb) * pred_chr
        final_preds[crop][h] = np.expm1(best_ensemble_log)

print("✅ 모든 품목 및 시점에 대한 메타 앙상블 가중치 최적화가 완료되었습니다.")


## 8. 결과 시각화 및 모델 평가

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 한글 폰트 설정 (필요시)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

def plot_final_results(crop, horizon):
    """
    특정 품목과 시점의 예측 결과를 시각화
    """
    if horizon not in final_preds.get(crop, {}):
        print(f"데이터 부족: {crop} - {horizon}일")
        return
        
    # 1. 시각화 데이터 준비
    item_test = test_df[test_df['품목'] == crop]
    y_true = np.expm1(item_test['평균가격'].shift(-horizon)).dropna()
    y_final = final_preds[crop][horizon]
    
    # 공통 기간 추출
    common_idx = y_true.index.intersection(y_final.index)
    if len(common_idx) == 0: return
    
    # 2. 그래프 그리기
    plt.figure(figsize=(12, 5))
    plt.plot(common_idx, y_true.loc[common_idx], label='Actual Price', color='black', linewidth=2, alpha=0.6)
    plt.plot(common_idx, y_final.loc[common_idx], label=f'Ensemble Pred (t+{horizon})', color='red', linestyle='--')
    
    # 가중치 정보 표시
    w_lgb = ensemble_weights[crop][horizon]
    plt.title(f"[{crop}] {horizon} Days Forecast - (LGBM Ratio: {w_lgb:.2f})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# --- 전체 성적 계산 ---
total_nmae_list = []
print(f"{'품목':<10} | {'7일 NMAE':<10} | {'14일 NMAE':<10} | {'28일 NMAE':<10}")
print("-" * 55)

for crop in crops:
    crop_nmaes = []
    line_str = f"{crop:<10} | "
    
    for h in horizons:
        if h in final_preds.get(crop, {}):
            item_test = test_df[test_df['품목'] == crop]
            y_true = np.expm1(item_test['평균가격'].shift(-h)).loc[final_preds[crop][h].index]
            score = nmae(y_true, final_preds[crop][h])
            crop_nmaes.append(score)
            line_str += f"{score:.4f}     | "
        else:
            line_str += f"{'N/A':<10} | "
            
    if crop_nmaes:
        total_nmae_list.append(np.mean(crop_nmaes))
    print(line_str)

print("-" * 55)
print(f"🏆 최종 평균 NMAE: {np.mean(total_nmae_list):.5f}")

# 주요 품목 시각화 예시 (상추, 배추 등)
plot_final_results('청상추', 7)
plot_final_results('배추', 28)


## 9. 최종 제출 파일 생성 (1월 1일~31일, 93행 생성)

In [ ]:
# 테스트 데이터 내 1월 한 달간의 날짜 범위 설정
test_year = test_df.index.year.unique()[0]
prediction_dates = pd.date_range(f'{test_year}-01-01', f'{test_year}-01-31')

submission_rows = []

for d in prediction_dates:
    for h in [7, 14, 28]:
        row = {'date': d.strftime('%Y-%m-%d'), 'horizon': f't+{h}'}
        for crop in crops:
            if h in final_preds[crop] and d in final_preds[crop][h].index:
                pred_real = np.expm1(final_preds[crop][h].loc[d])
            else:
                pred_real = 0
            row[f'{crop}_가격(원/kg)'] = pred_real
        submission_rows.append(row)

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv('sample_submission_ensemble.csv', index=False)
print(f"제출 파일 생성 완료: {len(submission_df)}개의 행이 생성되었습니다. (31일 x 3시점)")